In [ ]:
import io
import time
import pandas as pd
import requests
import calendar
from pathlib import Path
from datetime import date, timedelta

MAP_KEY = 'cbf821fc44c5cdb58ef790f8c1540286'
STATUS_URL = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY
BASE_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"
SOURCE = "MODIS_SP"
# Your HydroViewer/Amazon domain
AREA = "-82.0,-21.0,-49.0, 6.0"

OUT_DIR = Path("/mnt/e/backup/FIRMS_MODIS_DAILY_CSV")
OUT_DIR.mkdir(exist_ok=True)

try:
    response = requests.get(STATUS_URL)
    data = response.json()
    df = pd.Series(data)
    print(df)
except:
  print ("There is an issue with the query. \nTry in your browser: %s" % STATUS_URL)


def get_available_dates():
    url = (
        "https://firms.modaps.eosdis.nasa.gov/"
        f"api/data_availability/csv/{MAP_KEY}/{SOURCE}"
    )

    df = pd.read_csv(url)

    return (
        pd.Timestamp(df.iloc[0]["min_date"]),
        pd.Timestamp(df.iloc[0]["max_date"]),
    )

start_date, end_date = get_available_dates()

print("\n Available:", start_date, "->", end_date)

transaction_limit             5000
current_transactions             0
transaction_interval    10 minutes
dtype: object


In [14]:
def download_chunk(start_date, days=5):
    """Download one <=5-day FIRMS chunk."""

    url = (
        f"{BASE_URL}/{MAP_KEY}/{SOURCE}/"
        f"{AREA}/{days}/{start_date.isoformat()}"
    )

    print(f"Downloading {start_date} ({days} days)")

    r = requests.get(url, timeout=120)
    r.raise_for_status()

    # FIRMS may legitimately return no detections
    if not r.text.strip():
        return pd.DataFrame()

    return pd.read_csv(io.StringIO(r.text))

def download_month(year, month):
    """Download one complete month and save as one CSV."""

    _, n_days = calendar.monthrange(year, month)

    start_date = date(year, month, 1)
    end_date = date(year, month, n_days)

    outfile = OUT_DIR / f"{SOURCE}_{year}_{month:02d}.csv"

    # Makes script restartable
    if outfile.exists():
        print("Already exists:", outfile)
        return pd.read_csv(outfile)

    dfs = []
    current = start_date

    while current <= end_date:

        remaining = (end_date - current).days + 1
        days = min(5, remaining)

        try:
            df = download_chunk(current, days)

        except Exception as e:
            print("ERROR:", current, e)
            print("Retrying in 30 seconds...")

            time.sleep(30)

            try:
                df = download_chunk(current, days)

            except Exception as e:
                print("FAILED:", current, e)
                raise

        if not df.empty:
            dfs.append(df)
            print(f"  -> {len(df):,} detections")
        else:
            print("  -> no detections")

        # Don't hammer FIRMS
        time.sleep(1)

        current += timedelta(days=days)

    if not dfs:
        print(f"No detections for {year}-{month:02d}")
        return pd.DataFrame()

    # Combine 5-day requests into one monthly dataframe
    df = pd.concat(dfs, ignore_index=True)

    # Safety check: keep only requested calendar month
    df["acq_date"] = pd.to_datetime(df["acq_date"])

    df = df[
        (df["acq_date"].dt.year == year)
        & (df["acq_date"].dt.month == month)
    ]

    df.to_csv(outfile, index=False)

    print(
        f"Saved {outfile}: "
        f"{len(df):,} detections"
    )

    return df

In [ ]:
for year in range(2021, 2026):
    for month in range(1, 13):
        print(f"\n=== {year}-{month:02d} ===")

        try:
            download_month(year, month)
        except Exception as e:
            print(f"FAILED MONTH {year}-{month:02d}: {e}")